# AI-Assisted Shock Analysis using Synthetic ANSYS Data
This notebook combines:
1. Data Loading
2. EDA
3. Synthetic Transient Shock Signals
4. Random Forest, XGBoost, LightGBM, CatBoost
5. Neural Network
6. Hyperparameter Tuning
7. SHAP Explainability
8. Model Comparison

In [ ]:
# Install once if needed
# !pip install xgboost lightgbm catboost tensorflow shap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout
from tensorflow.keras.callbacks import EarlyStopping
import shap

df=pd.read_csv('synthetic_ansys_shock_data.csv')
display(df.head())
print(df.describe())


In [ ]:
# ---------------- EDA ----------------
print(df.info())
print(df.isnull().sum())

sns.heatmap(df.corr(),cmap='coolwarm')
plt.show()

df.hist(figsize=(12,10))
plt.tight_layout()
plt.show()

sns.pairplot(df[['Gun_g','FrameStress','CameraStress','CameraAcceleration','RMSStress']])
plt.show()

sns.boxplot(data=df[['FrameStress','CameraStress','RMSStress']]);plt.show()

plt.scatter(df['Gun_g'],df['CameraStress']);plt.xlabel('Gun g');plt.ylabel('Camera Stress');plt.show()

plt.scatter(df['Spring_k'],df['CameraAcceleration']);plt.show()

sns.kdeplot(df['FrameStress'],label='Frame')
sns.kdeplot(df['CameraStress'],label='Camera')
plt.legend();plt.show()

df.corr()['CameraStress'].sort_values().plot(kind='barh');plt.show()

df[['FrameStress','CameraStress','RMSStress','CameraAcceleration']].mean().plot(kind='bar');plt.show()


In [ ]:
# -------- Transient Shock Signal --------
t=np.linspace(0,0.05,1000)
s=df.iloc[0]
gun=s.Gun_g*np.exp(-120*t)*np.sin(2*np.pi*220*t)
frame=s.FrameStress*np.exp(-80*t)*np.sin(2*np.pi*180*t)
rms=s.RMSStress*np.exp(-60*t)*np.sin(2*np.pi*120*t)
cam=s.CameraAcceleration*np.exp(-40*t)*np.sin(2*np.pi*90*t)

plt.figure(figsize=(10,5))
plt.plot(t*1000,gun,label='Gun')
plt.plot(t*1000,frame,label='Frame')
plt.plot(t*1000,rms,label='RMS')
plt.plot(t*1000,cam,label='Camera')
plt.xlabel('Time (ms)')
plt.legend()
plt.grid()
plt.show()


In [ ]:
# -------- Machine Learning --------
X=df[['Gun_g','Gun_mass','Camera_mass','Spring_k','Damping','FrameThickness','Distance','YoungsModulus']]
Y=df[['FrameStress','FrameDeformation','RMSStress','CameraAcceleration','CameraStress','CameraDeformation']]
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

models={
'RF':MultiOutputRegressor(RandomForestRegressor(n_estimators=200,random_state=42)),
'XGB':MultiOutputRegressor(XGBRegressor(objective='reg:squarederror',n_estimators=200)),
'LGBM':MultiOutputRegressor(LGBMRegressor(n_estimators=200)),
'CAT':MultiOutputRegressor(CatBoostRegressor(iterations=200,verbose=False))
}
results=[]
for n,m in models.items():
    m.fit(X_train,Y_train)
    p=m.predict(X_test)
    results.append([n,r2_score(Y_test,p),mean_absolute_error(Y_test,p),np.sqrt(mean_squared_error(Y_test,p))])
results_df=pd.DataFrame(results,columns=['Model','R2','MAE','RMSE'])
display(results_df)


In [ ]:
# -------- Neural Network --------
sc=StandardScaler()
Xs=sc.fit_transform(X_train)
Xt=sc.transform(X_test)

nn=Sequential([
Dense(128,activation='relu',input_shape=(Xs.shape[1],)),
Dropout(0.2),
Dense(64,activation='relu'),
Dense(32,activation='relu'),
Dense(Y_train.shape[1])
])
nn.compile(optimizer='adam',loss='mse')
nn.fit(Xs,Y_train,epochs=50,batch_size=32,validation_split=0.2,
callbacks=[EarlyStopping(patience=8,restore_best_weights=True)],verbose=0)
pred=nn.predict(Xt,verbose=0)
results_df.loc[len(results_df)]=['NeuralNet',r2_score(Y_test,pred),mean_absolute_error(Y_test,pred),np.sqrt(mean_squared_error(Y_test,pred))]
display(results_df)


In [ ]:
# -------- Hyperparameter Tuning + SHAP --------
params={'n_estimators':[100,200,300],'max_depth':[5,10,None]}
search=RandomizedSearchCV(RandomForestRegressor(),params,n_iter=4,cv=3,random_state=42)
search.fit(X_train,Y_train)
best=search.best_estimator_
print(search.best_params_)

explainer=shap.TreeExplainer(best)
sample=X_test.iloc[:200]
sv=explainer.shap_values(sample)
shap.summary_plot(sv[0],sample,feature_names=X.columns)

pd.Series(best.feature_importances_,index=X.columns).sort_values().plot(kind='barh')
plt.show()

results_df.sort_values('R2').plot(x='Model',y='R2',kind='bar')
plt.show()
